In [ ]:
# cell 1
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
# cell 2
INPUT_FILE = Path(r"..\Data\processed_data\data_encoded.csv")
OUTPUT_DIR = Path(r"..\Data\model_data")

TEST_SIZE = 0.2
RANDOM_STATE = 10

OUTCOMES = {
    "mask": "face_mask_behaviour_binary",
    "protective": "protective_behaviour_binary",
    "social_avoidance": "social_avoidance_binary"
}

ID_COLUMNS = ["RecordNo", "endtime"]

In [ ]:
# cell 3
# Helper function
# summarize class balance
def make_class_balance_table(y, dataset_name, outcome_name):
    counts = y.value_counts().sort_index()
    proportions = y.value_counts(normalize=True).sort_index()

    return pd.DataFrame({
        "outcome": outcome_name,
        "dataset": dataset_name,
        "class": counts.index,
        "count": counts.values,
        "proportion": proportions.values
    })

# save function
def save_split_dataset(output_dir, prefix, X_train, X_test, y_train, y_test):
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train.to_csv(output_dir / f"{prefix}_X_train.csv", index=False)
    X_test.to_csv(output_dir / f"{prefix}_X_test.csv", index=False)

    y_train.to_csv(output_dir / f"{prefix}_y_train.csv", index=False)
    y_test.to_csv(output_dir / f"{prefix}_y_test.csv", index=False)


In [ ]:
# cell 4
# Load encoded data
df = pd.read_csv(INPUT_FILE)

print(f"Loaded dataset: {df.shape[0]} observations, {df.shape[1]} variables")


Loaded dataset: 40136 observations, 67 variables


In [ ]:
# cell 5
# Create outcome-specific datasets
all_outcome_cols = list(OUTCOMES.values())

balance_tables = []
metadata_rows = []

for prefix, target_col in OUTCOMES.items():

    print("\n" + "=" * 60)
    print(f"Preparing dataset for outcome: {prefix}")
    print(f"Target column: {target_col}")

    y = df[target_col].copy()

    drop_cols = ID_COLUMNS + all_outcome_cols
    X = df.drop(columns=drop_cols).copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    save_split_dataset(
        output_dir=OUTPUT_DIR,
        prefix=prefix,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    )

    balance_tables.append(make_class_balance_table(y, "full", prefix))
    balance_tables.append(make_class_balance_table(y_train, "train", prefix))
    balance_tables.append(make_class_balance_table(y_test, "test", prefix))

    metadata_rows.append({
        "outcome": prefix,
        "target_column": target_col,
        "n_full": len(y),
        "n_train": len(y_train),
        "n_test": len(y_test),
        "n_features": X.shape[1],
        "dropped_columns": ", ".join(drop_cols)
    })

    print(f"X shape: {X.shape}")
    print(f"Train shape: {X_train.shape}")
    print(f"Test shape: {X_test.shape}")

    print("Full class balance:")
    print(y.value_counts(normalize=True).sort_index())

    print("Train class balance:")
    print(y_train.value_counts(normalize=True).sort_index())

    print("Test class balance:")
    print(y_test.value_counts(normalize=True).sort_index())




Preparing dataset for outcome: mask
Target column: face_mask_behaviour_binary
X shape: (40136, 62)
Train shape: (32108, 62)
Test shape: (8028, 62)
Full class balance:
face_mask_behaviour_binary
0    0.460385
1    0.539615
Name: proportion, dtype: float64
Train class balance:
face_mask_behaviour_binary
0    0.460384
1    0.539616
Name: proportion, dtype: float64
Test class balance:
face_mask_behaviour_binary
0    0.460389
1    0.539611
Name: proportion, dtype: float64

Preparing dataset for outcome: protective
Target column: protective_behaviour_binary
X shape: (40136, 62)
Train shape: (32108, 62)
Test shape: (8028, 62)
Full class balance:
protective_behaviour_binary
0    0.34849
1    0.65151
Name: proportion, dtype: float64
Train class balance:
protective_behaviour_binary
0    0.34848
1    0.65152
Name: proportion, dtype: float64
Test class balance:
protective_behaviour_binary
0    0.34853
1    0.65147
Name: proportion, dtype: float64

Preparing dataset for outcome: social_avoidance
T

In [ ]:
# cell 6
# Save summary files
class_balance_summary = pd.concat(balance_tables, ignore_index=True)
model_dataset_metadata = pd.DataFrame(metadata_rows)

class_balance_summary.to_csv(
    OUTPUT_DIR / "class_balance_summary.csv",
    index=False
)

model_dataset_metadata.to_csv(
    OUTPUT_DIR / "model_dataset_metadata.csv",
    index=False
)

pd.Series(X.columns, name="feature").to_csv(
    OUTPUT_DIR / "feature_columns.csv",
    index=False
)